# Fase 0 — Análisis Out-of-Sample

Análisis completo de las 5 estrategias × 4 activos × 2 timeframes sobre el período OOS reservado.

> **Disclaimer:** Este notebook es parte de un experimento educativo. No constituye asesoramiento financiero.
> Todo el testing posterior se realiza en cuentas demo.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Localizar raíz del proyecto desde el notebook
root = Path.cwd()
if not (root / 'reports').exists():
    root = root.parent
sys.path.insert(0, str(root))

from backtest.engine import run_oos_backtest
from backtest.grids import ALL_STRATEGIES
from backtest.metrics import beats_buy_hold, passes_filters
from data.loader import OOS_END, OOS_START, load_split

STRATEGY_MAP = {cls.__name__: cls for cls in ALL_STRATEGIES}
SLIPPAGE = {'SPY': 0.0003, 'AAPL': 0.0005, 'NVDA': 0.0010, 'TSLA': 0.0015}
REPORTS = root / 'reports'

print(f'Raíz del proyecto: {root}')
print(f'Período OOS: {OOS_START} → {OOS_END}')

In [ ]:
# Cargar resultados OOS
df_oos = pd.read_parquet(REPORTS / 'fase0_oos.parquet')
df_wf  = pd.read_parquet(REPORTS / 'fase0_results.parquet')

print(f'OOS combos: {len(df_oos)}')
print(f'Walk-forward folds: {len(df_wf)}')
df_oos.head(3)

## 1. Tabla Resumen OOS

Métricas clave para cada combinación en el período OOS final.

In [ ]:
# Añadir degradación WF → OOS
wf_sharpe = (
    df_wf.dropna(subset=['sharpe'])
    .groupby(['strategy', 'ticker', 'timeframe'])['sharpe']
    .mean()
    .reset_index()
    .rename(columns={'sharpe': 'sharpe_wf'})
)
df_summary = df_oos.merge(wf_sharpe, on=['strategy', 'ticker', 'timeframe'], how='left')
df_summary['degradacion_pct'] = (
    (df_summary['sharpe'] - df_summary['sharpe_wf'])
    / df_summary['sharpe_wf'].abs() * 100
)

cols = ['strategy', 'ticker', 'timeframe',
        'sharpe', 'max_drawdown', 'profit_factor', 'num_trades',
        'total_return', 'buy_hold_return',
        'passes_filters', 'beats_buy_hold', 'degradacion_pct']
available = [c for c in cols if c in df_summary.columns]

display_df = (
    df_summary[available]
    .sort_values('sharpe', ascending=False)
    .reset_index(drop=True)
)

# Formatear para mejor lectura
fmt = {
    'sharpe':          '{:.3f}',
    'max_drawdown':    '{:.1f}%',
    'profit_factor':   '{:.2f}',
    'total_return':    '{:.1f}%',
    'buy_hold_return': '{:.1f}%',
    'degradacion_pct': '{:+.1f}%',
}
display(display_df.style.format(fmt, na_rep='N/A'))

In [ ]:
# Estadísticas de filtros
n_total = len(df_oos)
n_pass  = int(df_oos['passes_filters'].sum())
n_bh    = int(df_oos['beats_buy_hold'].sum())
n_both  = int((df_oos['passes_filters'] & df_oos['beats_buy_hold']).sum())

print(f'Total combos OOS evaluados : {n_total}')
print(f'Pasan los 4 filtros        : {n_pass}/{n_total}  ({100*n_pass/n_total:.1f}%)')
print(f'Superan buy & hold         : {n_bh}/{n_total}  ({100*n_bh/n_total:.1f}%)')
print(f'Pasan filtros Y B&H        : {n_both}/{n_total}  ({100*n_both/n_total:.1f}%)')
print()
print('Filtros aplicados al OOS FINAL completo:')
print('  Sharpe > 1.0 | Max Drawdown < 20% | Profit Factor > 1.3 | Num Trades >= 30')

## 2. Heatmap Sharpe OOS

In [ ]:
for tf in ['1d', '1h']:
    sub = df_oos[df_oos['timeframe'] == tf].copy()
    if sub.empty:
        continue

    pivot = sub.pivot_table(index='strategy', columns='ticker', values='sharpe', aggfunc='mean')

    fig = px.imshow(
        pivot,
        text_auto='.2f',
        color_continuous_scale='RdYlGn',
        color_continuous_midpoint=0,
        zmin=-2, zmax=2,
        title=f'Sharpe Ratio OOS — Timeframe {tf}',
        labels={'color': 'Sharpe'},
        aspect='auto',
    )
    fig.update_layout(height=350, font=dict(size=12))
    fig.show()

## 3. Curvas de Equity — Top 5 con Buy & Hold

In [ ]:
top5 = df_oos.nlargest(5, 'sharpe')[['strategy', 'ticker', 'timeframe', 'sharpe', 'total_return']]
print('Top 5 por Sharpe OOS:')
display(top5.reset_index(drop=True))

In [ ]:
import ast
from collections import Counter

def get_best_params(strategy_name, ticker, timeframe):
    group = df_wf[
        (df_wf['strategy']  == strategy_name) &
        (df_wf['ticker']    == ticker) &
        (df_wf['timeframe'] == timeframe)
    ]['best_params'].dropna().tolist()
    if not group:
        return {}
    most_common = Counter(group).most_common(1)[0][0]
    try:
        return ast.literal_eval(most_common)
    except Exception:
        return {}

fig = make_subplots(
    rows=5, cols=1,
    subplot_titles=[
        f"{r['strategy']} | {r['ticker']} {r['timeframe']} "
        f"(Sharpe: {r['sharpe']:.2f})"
        for _, r in top5.iterrows()
    ],
    vertical_spacing=0.06,
)

for i, (_, row) in enumerate(top5.iterrows(), 1):
    try:
        strategy_cls = STRATEGY_MAP[row['strategy']]
        _, df_oos_data = load_split(row['ticker'], row['timeframe'])
        best_params = get_best_params(row['strategy'], row['ticker'], row['timeframe'])
        slippage = SLIPPAGE.get(row['ticker'], 0.0005)

        _, _, equity_df = run_oos_backtest(
            strategy_cls, df_oos_data, params=best_params,
            cash=10_000, slippage=slippage,
        )

        # Curva equity de la estrategia
        if not equity_df.empty and 'Equity' in equity_df.columns:
            fig.add_trace(
                go.Scatter(
                    x=equity_df.index, y=equity_df['Equity'],
                    name=f"{row['strategy']} {row['ticker']}",
                    line=dict(color='steelblue', width=2),
                    showlegend=(i == 1),
                ),
                row=i, col=1,
            )

        # Línea buy & hold (normalizada al mismo capital inicial)
        if not df_oos_data.empty:
            bh = 10_000 * (df_oos_data['Close'] / df_oos_data['Close'].iloc[0])
            fig.add_trace(
                go.Scatter(
                    x=df_oos_data.index, y=bh,
                    name='Buy & Hold',
                    line=dict(color='orange', dash='dash', width=1.5),
                    showlegend=(i == 1),
                ),
                row=i, col=1,
            )

    except Exception as e:
        print(f'  WARN: error graficando {row["strategy"]} {row["ticker"]}: {e}')

fig.update_layout(
    height=300 * 5,
    title_text='Curvas de Equity OOS — Top 5 (azul=estrategia, naranja=buy&hold)',
    showlegend=True,
)
fig.show()

## 4. Distribución de Retornos por Trade

In [ ]:
trades_path = REPORTS / 'fase0_oos_trades.parquet'
if trades_path.exists():
    df_trades = pd.read_parquet(trades_path)
    print(f'Total trades OOS: {len(df_trades)}')
    display(df_trades.head(3))
else:
    print('Archivo de trades no disponible — se omite esta sección.')
    df_trades = pd.DataFrame()

In [ ]:
if not df_trades.empty and 'ReturnPct' in df_trades.columns:
    # Top 5 combos para comparar distribuciones
    top5_keys = list(zip(top5['strategy'], top5['ticker'], top5['timeframe']))

    fig = go.Figure()
    for strat, ticker, tf in top5_keys:
        sub = df_trades[
            (df_trades['strategy']  == strat) &
            (df_trades['ticker']    == ticker) &
            (df_trades['timeframe'] == tf)
        ]['ReturnPct'] * 100

        if sub.empty:
            continue
        fig.add_trace(go.Histogram(
            x=sub,
            name=f'{strat[:12]} {ticker} {tf}',
            opacity=0.6,
            nbinsx=30,
        ))

    fig.add_vline(x=0, line_dash='dash', line_color='red', annotation_text='0%')
    fig.update_layout(
        barmode='overlay',
        title='Distribución de retornos por trade (Top 5, OOS)',
        xaxis_title='Retorno por trade (%)',
        yaxis_title='Frecuencia',
        height=450,
    )
    fig.show()

    # Estadísticas descriptivas
    for strat, ticker, tf in top5_keys:
        sub = df_trades[
            (df_trades['strategy']  == strat) &
            (df_trades['ticker']    == ticker) &
            (df_trades['timeframe'] == tf)
        ]['ReturnPct'] * 100
        if sub.empty:
            continue
        print(f'{strat} {ticker} {tf}: '
              f'n={len(sub)}, media={sub.mean():.2f}%, '
              f'mediana={sub.median():.2f}%, '
              f'win_rate={100*(sub>0).mean():.1f}%')
else:
    print('Sin datos de trades para graficar.')

## 5. Drawdowns — Duración y Momento

In [ ]:
# Para top 3 — graficar drawdown temporal
top3 = top5.head(3)

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=[
        f"Drawdown — {r['strategy']} {r['ticker']} {r['timeframe']}"
        for _, r in top3.iterrows()
    ],
    vertical_spacing=0.08,
)

for i, (_, row) in enumerate(top3.iterrows(), 1):
    try:
        strategy_cls = STRATEGY_MAP[row['strategy']]
        _, df_oos_data = load_split(row['ticker'], row['timeframe'])
        best_params = get_best_params(row['strategy'], row['ticker'], row['timeframe'])
        slippage = SLIPPAGE.get(row['ticker'], 0.0005)

        _, _, equity_df = run_oos_backtest(
            strategy_cls, df_oos_data, params=best_params,
            cash=10_000, slippage=slippage,
        )

        if not equity_df.empty and 'DrawdownPct' in equity_df.columns:
            fig.add_trace(
                go.Scatter(
                    x=equity_df.index,
                    y=-equity_df['DrawdownPct'] * 100,
                    fill='tozeroy',
                    fillcolor='rgba(255,50,50,0.3)',
                    line=dict(color='red', width=1),
                    name=f"{row['strategy']} {row['ticker']}",
                    showlegend=False,
                ),
                row=i, col=1,
            )
    except Exception as e:
        print(f'  WARN: {e}')

fig.update_layout(
    height=300 * 3,
    title_text='Drawdowns OOS (área roja = pérdida respecto al máximo)',
)
fig.update_yaxes(title_text='Drawdown (%)')
fig.show()

## 6. Degradación Walk-Forward → OOS

Comparación del Sharpe medio en los folds de test del walk-forward vs el Sharpe en el OOS final.
Una degradación >30% sugiere posible overfitting durante el proceso de optimización.

In [ ]:
wf_sharpe = (
    df_wf.dropna(subset=['sharpe'])
    .groupby(['strategy', 'ticker', 'timeframe'])['sharpe']
    .mean()
    .reset_index()
    .rename(columns={'sharpe': 'sharpe_wf'})
)

df_deg = df_oos.merge(wf_sharpe, on=['strategy', 'ticker', 'timeframe'], how='left')
df_deg['degradacion_pct'] = (
    (df_deg['sharpe'] - df_deg['sharpe_wf'])
    / df_deg['sharpe_wf'].abs() * 100
)

fig = go.Figure()

colors = df_deg['degradacion_pct'].apply(
    lambda x: 'red' if x < -30 else ('orange' if x < 0 else 'green')
)

for tf in ['1d', '1h']:
    sub = df_deg[df_deg['timeframe'] == tf]
    if sub.empty:
        continue
    labels = sub['strategy'].str[:10] + ' ' + sub['ticker']
    fig.add_trace(go.Bar(
        x=labels,
        y=sub['degradacion_pct'],
        name=tf,
        marker_color=sub['degradacion_pct'].apply(
            lambda x: 'crimson' if x < -30 else ('darkorange' if x < 0 else 'seagreen')
        ),
    ))

fig.add_hline(y=-30, line_dash='dash', line_color='red',
              annotation_text='Umbral sospecha (-30%)', annotation_position='top right')
fig.add_hline(y=0, line_color='black', line_width=1)
fig.update_layout(
    title='Degradación Sharpe: Walk-Forward (test folds) → OOS Final',
    yaxis_title='Degradación (%)',
    xaxis_title='Combo',
    height=500,
    xaxis_tickangle=45,
)
fig.show()

# Combos con degradación crítica
critical = df_deg[df_deg['degradacion_pct'] < -30]
print(f'Combos con degradación >30% (posible overfitting): {len(critical)}/{len(df_deg)}')
if not critical.empty:
    print(critical[['strategy', 'ticker', 'timeframe', 'sharpe_wf', 'sharpe', 'degradacion_pct']]
          .sort_values('degradacion_pct').to_string(index=False))

## 7. Veredicto Final

In [ ]:
print('='*70)
print('VEREDICTO FINAL — FASE 0')
print('='*70)
print()
print(f'Período OOS evaluado: {OOS_START} → {OOS_END}')
print(f'Total combinaciones : {n_total}')
print()
print(f'Pasan los 4 filtros (Sharpe>1, DD<20%, PF>1.3, Trades≥30): {n_pass}/{n_total}')
print(f'Superan buy & hold en el mismo período                    : {n_bh}/{n_total}')
print(f'Pasan filtros Y superan buy & hold                        : {n_both}/{n_total}')
print()

if n_both > 0:
    cands = df_oos[df_oos['passes_filters'] & df_oos['beats_buy_hold']]
    print(f'CANDIDATOS PARA FASE 1 (paper trading):')
    display(cands[['strategy', 'ticker', 'timeframe', 'sharpe',
                   'max_drawdown', 'profit_factor', 'total_return', 'buy_hold_return']]
            .sort_values('sharpe', ascending=False))
elif n_pass > 0:
    print('Alguna estrategia pasa filtros pero ninguna supera buy & hold.')
    print('No hay candidatos claros para Fase 1.')
else:
    print('Ninguna estrategia pasa los 4 filtros en OOS.')
    print('Resultado válido: estas estrategias no tienen ventaja estadística')
    print('suficiente en este período y activos.')
    print()
    print('Ver RECOMENDACION.md para próximos pasos.')

print()
print('Ver reports/RECOMENDACION.md para el análisis completo.')